In [0]:
import logging
import random
import re
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

# ── Logging setup ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("week2")

# ── Plot defaults ──────────────────────────────────────────────────────────────
plt.rcParams["figure.dpi"]        = 110
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

# ── Catalog / schema constants ─────────────────────────────────────────────────
CATALOG = "sandbox_catalog"
SCHEMA  = "banking_details"

log.info("Imports complete. Catalog=%s  Schema=%s", CATALOG, SCHEMA)

18:51:57  INFO  Imports complete. Catalog=sandbox_catalog  Schema=banking_details


In [0]:
# Create the schema (database) inside our catalog if it does not already exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")



18:52:10  INFO  Closing down clientserver connection


DataFrame[]

In [0]:
#Customer table
PROVINCE = [
    "Alberta", "AB", "alberta",
    "British Columbia", "BC", "british columbia",
    "Manitoba", "MB", "manitoba",
    "New Brunswick", "NB", "new brunswick",
    "Newfoundland and Labrador", "NL", "newfoundland and labrador",
    "Nova Scotia", "NS", "nova scotia",
    "Ontario", "ON", "ontario",
    "Prince Edward Island", "PE", "prince edward island",
    "Quebec", "QC", "quebec",
    "Saskatchewan", "SK", "saskatchewan"
]

customers_data = []
for i in range(1000):
    join_date = (datetime.now() - timedelta(days=random.randint(0, 365*10))).date()
    customers_data.append((i, random.randint(18, 70), random.choice(PROVINCE), random.choice(["$0-$25K", "$25K-$50K", "$50K-$75K", "$75K-$100K", "$100K-$150K", "$150K-$200K", "$200K+"]), join_date))

spark.createDataFrame(customers_data, ["customer_id", "age", "province", "income_bracket", "join_date"]).write.mode("overwrite").saveAsTable("customers")
display(spark.sql("select * from customers"))

customer_id,age,province,income_bracket,join_date
0,43,prince edward island,$200K+,2017-05-31
1,68,newfoundland and labrador,$50K-$75K,2022-12-11
2,69,newfoundland and labrador,$150K-$200K,2026-03-24
3,70,NL,$50K-$75K,2020-07-25
4,20,Quebec,$100K-$150K,2017-03-23
5,39,NS,$100K-$150K,2017-06-20
6,48,saskatchewan,$25K-$50K,2018-10-08
7,42,Quebec,$200K+,2019-01-16
8,41,PE,$200K+,2019-03-23
9,51,Ontario,$50K-$75K,2017-05-01


In [0]:
#Accounts table
ACCOUNT_TYPE= ['savings', 'CHECKINGS', 'Savings', 'Checkings']
STATUS = ['active', 'inactive', 'closed']

accounts_data= []

for i in range(1000):
    join_date = (datetime.now() - timedelta(days=random.randint(0, 365*10))).date()
    accounts_data.append((i, i, random.choice(ACCOUNT_TYPE),random.choice(STATUS), join_date, random.randint(1000, 1000000)))

spark.createDataFrame(accounts_data, ["account_id","customer_id", "account_type", "status", "open_date", "balance"]).write.mode("overwrite").saveAsTable("accounts")
display(spark.sql("select * from accounts"))


account_id,customer_id,account_type,status,open_date,balance
0,0,Savings,inactive,2020-12-13,919399
1,1,Checkings,inactive,2025-06-05,290685
2,2,savings,active,2021-05-20,293805
3,3,savings,active,2022-01-03,27300
4,4,CHECKINGS,closed,2025-09-04,880499
5,5,Savings,inactive,2017-11-20,709986
6,6,Checkings,closed,2021-03-27,176408
7,7,Savings,active,2016-08-15,278062
8,8,Checkings,closed,2021-06-03,89310
9,9,savings,active,2025-07-06,800709


In [0]:
# Transactions table
TRANSACTION_TYPE = ["deposit", "withdrawal", "transfer"]
TRANSACTION_WEIGHTS = [30, 50, 20]  

MERCHANT_CATEGORIES = ["Groceries", "Dining", "Gas & Transport", 
                        "Healthcare", "Entertainment", "Utilities", 
                        "Online Shopping", "Travel", "ATM Withdrawal"]

ACCOUNT_IDS = list(range(0, 1000))

START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 12, 31)

transactions_data = []

for i in range(10000): 
    account_id = random.choice(ACCOUNT_IDS)

    # Weighted transaction type
    transaction_type = random.choices(TRANSACTION_TYPE, weights=TRANSACTION_WEIGHTS)[0]

    random_days = random.randint(0, (END_DATE - START_DATE).days)
    transaction_date = (START_DATE + timedelta(days=random_days)).strftime("%Y-%m-%d")

    if random.random() < 0.03:
        amount = None
    elif transaction_type == "withdrawal" and random.random() < 0.05:
        amount = str(round(random.uniform(-500, -1), 2))
    else:
        amount = str(round(random.uniform(1, 5000), 2))

    if transaction_type in ("deposit", "transfer"):
        merchant_category = None
    elif random.random() < 0.05:
        merchant_category = None
    else:
        merchant_category = random.choice(MERCHANT_CATEGORIES)

    transactions_data.append((
        i, account_id, transaction_type, transaction_date, amount, merchant_category
    ))

spark.createDataFrame(
    transactions_data, 
    ["transaction_id", "account_id", "transaction_type", "transaction_date", "amount", "merchant_category"]
).write.mode("overwrite").saveAsTable("transactions")

display(spark.sql("select * from transactions"))

18:52:58  INFO  Received command c on object id p0


transaction_id,account_id,transaction_type,transaction_date,amount,merchant_category
0,487,deposit,2024-10-22,2421.75,null
1,323,withdrawal,2024-10-29,1458.29,ATM Withdrawal
2,562,withdrawal,2024-04-27,1542.41,Healthcare
3,803,transfer,2024-08-26,1567.89,null
4,263,deposit,2024-02-19,678.55,null
5,27,withdrawal,2024-01-11,648.53,Online Shopping
6,990,withdrawal,2024-01-03,2779.92,Utilities
7,123,withdrawal,2024-09-13,4765.63,Utilities
8,603,transfer,2024-07-06,1347.2,null
9,538,deposit,2024-02-28,3133.16,null


In [0]:
# Loan Applications table
LOAN_TYPE = ["mortgage", "auto", "personal"]
LOAN_STATUS = ["approved", "denied", "pending"]
STATUS_WEIGHTS = [50, 35, 15]

LOAN_AMOUNT_RANGES = {
    "mortgage": (100000, 900000),
    "auto": (5000, 80000),
    "personal": (1000, 50000)
}

INTEREST_RATE_RANGES = {
    "mortgage": (3.0, 7.0),
    "auto": (4.0, 12.0),
    "personal": (6.0, 25.0)
}

CUSTOMER_IDS = list(range(0, 1000))  # match your customers table

START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 12, 31)

loan_data = []

for i in range(600):
    customer_id = random.choice(CUSTOMER_IDS)
    loan_type = random.choice(LOAN_TYPE)

    # Application date stored as string (cast to date in Silver)
    random_days = random.randint(0, (END_DATE - START_DATE).days)
    application_date = (START_DATE + timedelta(days=random_days)).strftime("%Y-%m-%d")

    # Amount requested
    if random.random() < 0.05:
        amount_requested = None
    else:
        low, high = LOAN_AMOUNT_RANGES[loan_type]
        amount_requested = round(random.uniform(low, high), 2)

    # Status
    status = random.choices(LOAN_STATUS, weights=STATUS_WEIGHTS)[0]

    # Interest rate 
    if random.random() < 0.03:
        interest_rate = round(random.uniform(100.0, 200.0), 2)  # clearly invalid
    else:
        low, high = INTEREST_RATE_RANGES[loan_type]
        interest_rate = round(random.uniform(low, high), 2)

    loan_data.append((
        i, customer_id, loan_type, amount_requested, status, application_date, interest_rate
    ))

spark.createDataFrame(
    loan_data,
    ["application_id", "customer_id", "loan_type", "amount_requested", "status", "application_date", "interest_rate"]
).write.mode("overwrite").saveAsTable("loan_applications")

display(spark.sql("select * from loan_applications"))

application_id,customer_id,loan_type,amount_requested,status,application_date,interest_rate
0,429,personal,42177.48,pending,2024-01-18,14.5
1,182,mortgage,488145.02,pending,2024-05-06,6.11
2,599,personal,21850.92,approved,2024-02-17,24.06
3,469,mortgage,160401.15,approved,2024-05-11,4.28
4,467,personal,39834.72,denied,2024-10-23,16.0
5,956,auto,53736.99,denied,2024-07-17,7.1
6,513,personal,30068.3,approved,2024-02-02,6.72
7,495,mortgage,null,denied,2024-04-24,4.6
8,49,personal,32614.36,denied,2024-05-25,10.97
9,355,mortgage,664386.92,approved,2024-04-15,3.71
